# INF201 - Deliverable 2: Campus Coffee Shop
**Max Carl Andresen (maand1153) and Sam Mirtaheri (samir9125)**

We read cash-register logs, clean prices and produce reports using NumPy and pandas.
Tasks 1-4 define the functions; Task 5 combines them into one workflow.

**GitHub:** https://github.com/madmax502/inf201-h26-deliverables

Select the Python environment in `Deliverable_2/.venv` and run all cells from the top.
The input files in `data/` were generated with the supplied `generate_data_for_d2.py`.
Only run the generator again if you want a new random dataset. The four required
reports are written to `output/`.

## Collaboration
We meet in person and write the code together on one computer. Both group
members contribute to solving the tasks. Because we use one computer and Git
account, the commits are recorded under one person even though the work is shared.


In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd


## Task 1 - File ingestion and regular expressions
The reader tries UTF-8 first and falls back to Latin-1 if decoding fails.
It returns both the text and the encoding used. The regular expression extracts
the six transaction fields and ignores system-noise lines. Prices are converted
to `float`.


In [2]:
def read_file(file_path):
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    # Read as UTF-8, or use Latin-1 if decoding fails
    try:
        return path.read_text(encoding="utf-8"), "utf-8"
    except UnicodeDecodeError:
        return path.read_text(encoding="iso-8859-1"), "iso-8859-1"


In [3]:
LOG_PATTERN = re.compile(
    r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s+"
    r"\[(?P<register_id>[A-Za-z0-9_]+)\]\s+"
    r"item=(?P<item>.+?)\s+"
    r"price=(?P<price>\d+(?:\.\d+)?)\s+"
    r"(?P<currency>[A-Z]{3})\s+"
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)\s*$"
)


def parse_log(text):
    records = []
    # Extract transactions and skip system-noise lines
    for line in text.splitlines():
        match = LOG_PATTERN.match(line.strip())
        if match:
            record = match.groupdict()
            record["price"] = float(record["price"])
            records.append(record)
    return records


## Task 2 - Recursive utility functions
The directory search uses `Path.iterdir()` and explicit recursion to find log files
in nested folders, without `glob`, `rglob` or `os.walk`. The configuration reader
recursively visits dictionaries and lists and returns a set of active register IDs.


In [4]:
def find_all_logfiles(directory):
    path = Path(directory)
    if not path.is_dir():
        raise NotADirectoryError(f"Not a directory: {path}")

    logfiles = []
    # Search subfolders and collect log files
    for item in path.iterdir():
        if item.is_file() and item.suffix == ".log":
            logfiles.append(item)
        elif item.is_dir():
            logfiles.extend(find_all_logfiles(item))

    return sorted(logfiles)


In [5]:
def find_active_registers(data) -> set[str]:
    active_registers = set()

    # Traverse both dictionaries and lists
    if isinstance(data, dict):
        if data.get("active") is True and "id" in data:
            active_registers.add(data["id"])
        for value in data.values():
            active_registers.update(find_active_registers(value))

    elif isinstance(data, list):
        for item in data:
            active_registers.update(find_active_registers(item))

    return active_registers


## Task 3 - Cleaning and refund ratios with NumPy
Prices more than three standard deviations from the mean are replaced with the
median of the non-outlier prices. Cleaning takes place before the NOK correction,
as specified in Task 5.

The element-wise refund ratio is the number of refunds divided by the number of
successful sales plus `1e-6`. The small addition prevents division by zero.


In [6]:
def remove_price_outliers(prices):
    prices = np.asarray(prices, dtype=float).ravel()
    if prices.size == 0:
        return prices

    mean_price = prices.mean()
    std_dev = prices.std()

    # Find outliers and the median of the valid prices
    outliers = np.abs(prices - mean_price) > 3 * std_dev
    median_valid = np.median(prices[~outliers])

    cleaned = prices.copy()
    cleaned[outliers] = median_valid
    return cleaned


In [7]:
def compute_refund_ratio(
    sales_count: np.ndarray, refund_count: np.ndarray
) -> np.ndarray:
    sales_count = np.asarray(sales_count, dtype=float)
    refund_count = np.asarray(refund_count, dtype=float)
    return refund_count / (sales_count + 1e-6)


## Task 4 - Reports with pandas
Transactions are sorted by timestamp, which becomes the DataFrame index.
We sum prices per item and hour and create a pivot table by item and status.
Refund ratios use the counts of `REFUND` and `SUCCESS`, rather than their prices.

The hourly report sums all statuses to follow the requested `total price (sum)`.
It is not net revenue. The assignment specifies only a NOK price correction and
provides no exchange rates: the amounts are not converted to a common currency
and should not be interpreted as comparable financial revenue.


In [8]:
def transactions_to_dataframe(records):
    df = pd.DataFrame(records)
    df["timestamp"] = pd.to_datetime(
        df["timestamp"], format="%Y-%m-%d %H:%M:%S")
    df = df.set_index("timestamp")
    return df.sort_index()


In [9]:
def hourly_sales_by_item(df):
    # Sum all statuses as requested for total transaction prices
    return df.groupby("item").resample("1h")["price"].sum()


In [10]:
def revenue_pivot_by_status(df):
    pivot = pd.pivot_table(
        df, index="item", columns="status", values="price",
        aggfunc="sum", fill_value=0,
    )
    return pivot


In [11]:
def calc_refund_ratio_per_item(df):
    # Count successful sales and refunds per item
    counts = df.groupby(["item", "status"]).size().unstack(fill_value=0)
    counts = counts.reindex(columns=["SUCCESS", "REFUND"], fill_value=0)
    counts["refund_ratio"] = compute_refund_ratio(
        counts["SUCCESS"].to_numpy(), counts["REFUND"].to_numpy()
    )
    return counts[["refund_ratio"]]


## Task 5 - Complete workflow
We find the log files, read the configuration and parse transactions, then clean
prices and generate the reports. Active register IDs are extracted and displayed;
the assignment does not ask us to filter transactions using these IDs.


In [12]:
# Find log files and active registers
logfiles = find_all_logfiles("data")
config_text, _ = read_file("data/store_config.json")
config = json.loads(config_text)
active_registers = find_active_registers(config)

print(f"Number of log files: {len(logfiles)}")
print(f"Active registers: {', '.join(sorted(active_registers))}")


Number of log files: 9
Active registers: POS_01, POS_02, POS_03


In [13]:
# Read log files and extract transactions
records = []
for logfile in logfiles:
    text, encoding = read_file(logfile)
    records.extend(parse_log(text))

print(f"Number of transactions: {len(records)}")


Number of transactions: 720


In [14]:
# Create a DataFrame sorted by timestamp
df = transactions_to_dataframe(records)
print(df.head())


                    register_id        item  price currency   status
timestamp                                                           
2026-10-15 07:00:11      POS_02  Cappuccino   3.87      NOK  SUCCESS
2026-10-15 07:00:21      POS_03   Croissant   3.39      USD  SUCCESS
2026-10-15 07:01:11      POS_03   Croissant   3.97      EUR  SUCCESS
2026-10-15 07:01:13      POS_03   Croissant   4.05      NOK  SUCCESS
2026-10-15 07:01:17      POS_02   Croissant   3.90      USD  SUCCESS


In [15]:
# Replace price outliers with the median of valid prices
df["price"] = remove_price_outliers(df["price"])
print("Price statistics after cleaning, before NOK correction:")
print(df["price"].describe())


Price statistics after cleaning, before NOK correction:
count    720.000000
mean       4.446764
std        1.501844
min        2.510000
25%        3.467500
50%        4.010000
75%        4.612500
max        8.000000
Name: price, dtype: float64


In [16]:
# Correct NOK prices by multiplying them by ten
df.loc[df["currency"] == "NOK", "price"] *= 10.0


In [17]:
# Generate hourly totals, the pivot table and refund ratios
hourly_report = hourly_sales_by_item(df)
pivot_table = revenue_pivot_by_status(df)
refund_ratios = calc_refund_ratio_per_item(df)

print("Total price per item and hour:")
print(hourly_report.to_string())
print("\nTotal price per item and status:")
print(pivot_table.to_string(float_format="%.2f"))
print("\nRefund ratio per item:")
print(refund_ratios.to_string(float_format="%.4f"))


Total price per item and hour:
item        timestamp          
Cappuccino  2026-10-15 07:00:00     913.57
            2026-10-15 08:00:00    1104.95
            2026-10-15 09:00:00     142.06
Croissant   2026-10-15 07:00:00     804.84
            2026-10-15 08:00:00     869.21
            2026-10-15 09:00:00      46.12
Espresso    2026-10-15 07:00:00     723.00
            2026-10-15 08:00:00     996.97
            2026-10-15 09:00:00      44.41
Latte       2026-10-15 07:00:00    1111.23
            2026-10-15 08:00:00     874.66
            2026-10-15 09:00:00      96.22
Muffin      2026-10-15 07:00:00     651.67
            2026-10-15 08:00:00     442.57
            2026-10-15 09:00:00     150.24
Sandwich    2026-10-15 07:00:00    1757.59
            2026-10-15 08:00:00    2462.77
            2026-10-15 09:00:00     116.77

Total price per item and status:
status      CANCELLED  REFUND  SUCCESS
item                                  
Cappuccino     400.07  314.04  1446.47
Croissant   

In [18]:
# Save the four output files using the required filenames
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

df.to_csv(output_dir / "cleaned_transactions.csv", index_label="timestamp")
hourly_report.rename("total_price").reset_index().to_json(
    output_dir / "hourly_revenue.json",
    orient="records", date_format="iso", indent=2,
)
pivot_table.to_csv(output_dir / "status_pivot_table.csv")
refund_ratios.to_csv(output_dir / "item_refund_ratios.csv")


## Discussion of the results
We found 9 log files and parsed 720 transactions. The reader ignores system
messages, so these do not become rows in the reports. All register IDs in the logs
are included in the active-ID set from the configuration.

The three-standard-deviation rule identified 32 price outliers. After replacing
them, prices ranged from 2.51 to 8.00 before the NOK correction. Correcting NOK
prices then multiplies those rows by ten; a larger price at this stage is not
necessarily another hardware glitch.

Muffin had the highest refund-to-sale count ratio (25.0%), while
Espresso had the lowest (15.9%). These ratios compare refunds with
successful sales, not with all transactions. The hourly totals are much lower in
the final hour because the generated logs cover only part of that hour.

The totals combine currencies and include cancelled and refunded transactions.
They follow the assignment's requested sums, but would need exchange rates and
a clear treatment of refunds and cancellations before use as business revenue.


## Sources and AI use
- INF201: *Deliverable 2 - Campus Coffee Shop Sales Pipeline*.
- The supplied `generate_data_for_d2.py` script was used to generate the inputs.
- AI contributions are described in `ai_bruk.md`.
